# Часть 0 — самый простой MCP-сервер, прямо в блокноте

**MCP (Model Context Protocol)** — общий стандарт, как агенту подключаться к внешним инструментам и данным. Есть две стороны:

- **сервер** — программа, которая что-то умеет (сложить числа, сходить в базу, прочитать файл);
- **клиент** — тот, кто этим пользуется (Claude Desktop, Cursor, наш агент на LangGraph).

Они общаются JSON-сообщениями. Набор сообщений одинаковый для всех серверов в мире — поэтому любой клиент работает с любым сервером.

В этом блокноте сервер и клиент живут **в одном блокноте**: создаём сервер, тут же его дёргаем и видим каждое сообщение, которое между ними летает. Никаких баз, процессов и портов.

## 1. Сервер — это обычные функции с декораторами

Сервер умеет отдавать три вида вещей:

| что | декоратор | смысл | пример |
|---|---|---|---|
| **tool** (инструмент) | `@mcp.tool()` | действие, которое вызывает **модель** | сложить числа, узнать погоду |
| **resource** (ресурс) | `@mcp.resource(uri)` | данные «для чтения», их берёт **приложение** | описание сервера, схема БД |
| **prompt** (шаблон) | `@mcp.prompt()` | готовая заготовка запроса, её выбирает **пользователь** | «спланируй поездку в …» |

In [ ]:
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo", log_level="WARNING")   # the server; WARNING = no per-request logs


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@mcp.tool()
def get_weather(city: str) -> str:
    """Current weather in a city (fake data)."""
    return f"В городе {city} +18°C, солнечно"


@mcp.resource("demo://about")
def about() -> str:
    """What this server is."""
    return "Учебный MCP-сервер: складывает числа и сочиняет погоду."


@mcp.prompt()
def trip(city: str) -> str:
    """Plan a weekend trip."""
    return f"Спланируй выходные в городе {city}. Сначала узнай погоду."

Всё. Это готовый MCP-сервер. Никакого кода про протокол мы не писали — его берёт на себя **FastMCP** (как FastAPI берёт на себя HTTP).

## 2. Что FastMCP сделал из наших функций

Спросим у сервера список инструментов — ровно это потом увидит модель:

In [4]:
mcp.list_tools()

<coroutine object FastMCP.list_tools at 0x112d7f760>

In [5]:
import json

for tool in await mcp.list_tools():
    print(json.dumps(tool.model_dump(include={"name", "description", "inputSchema"}),
                     ensure_ascii=False, indent=2))

{
  "name": "add",
  "description": "Add two numbers.",
  "inputSchema": {
    "properties": {
      "a": {
        "title": "A",
        "type": "integer"
      },
      "b": {
        "title": "B",
        "type": "integer"
      }
    },
    "required": [
      "a",
      "b"
    ],
    "title": "addArguments",
    "type": "object"
  }
}
{
  "name": "get_weather",
  "description": "Current weather in a city (fake data).",
  "inputSchema": {
    "properties": {
      "city": {
        "title": "City",
        "type": "string"
      }
    },
    "required": [
      "city"
    ],
    "title": "get_weatherArguments",
    "type": "object"
  }
}


Откуда что взялось:

| в описании | из кода |
|---|---|
| `name: "add"` | имя функции |
| `description` | docstring |
| `inputSchema` (JSON Schema) | аннотации типов `a: int, b: int` → `"type": "integer"`, оба обязательны |

**Модель видит только это описание**, а не код функции. Поэтому хорошие имена и docstring — это и есть «промпт» для инструмента.

## 3. Подключаемся клиентом и смотрим на «провод»

Обычно сервер и клиент — разные программы, а между ними «провод»: stdin/stdout процесса или HTTP. Здесь мы соединим их прямо в памяти и поставим на провод «прослушку» — она печатает каждое сообщение:

- `→ клиент:` — что клиент отправил серверу;
- `← сервер:` — что сервер ответил.

Следующую ячейку можно не разбирать — это просто провод с прослушкой.

In [6]:
from contextlib import asynccontextmanager

import anyio
from mcp import ClientSession
from mcp.shared.memory import create_client_server_memory_streams


class Tap:
    """A wire that prints every JSON-RPC message passing through it."""

    def __init__(self, stream, arrow: str, state: dict) -> None:
        self.stream, self.arrow, self.state = stream, arrow, state

    async def send(self, item) -> None:
        if self.state["on"]:
            raw = item.message.model_dump_json(by_alias=True, exclude_none=True)
            print(self.arrow, raw if len(raw) < 400 else raw[:400] + " …")
        await self.stream.send(item)

    async def aclose(self) -> None:
        await self.stream.aclose()

    async def __aenter__(self):
        return self

    async def __aexit__(self, *exc) -> None:
        await self.aclose()


@asynccontextmanager
async def connect(server: FastMCP, show_handshake: bool = False):
    """Run `server` in memory and yield a client session connected to it."""
    state = {"on": show_handshake}
    async with create_client_server_memory_streams() as ((client_read, client_write), (server_read, server_write)):
        low_level = server._mcp_server
        async with anyio.create_task_group() as tasks:
            tasks.start_soon(lambda: low_level.run(server_read, Tap(server_write, "← сервер:", state),
                                                   low_level.create_initialization_options()))
            async with ClientSession(client_read, Tap(client_write, "→ клиент:", state)) as session:
                await session.initialize()
                state["on"] = True
                yield session
            tasks.cancel_scope.cancel()

### Рукопожатие и список инструментов

Первым делом клиент и сервер знакомятся (`initialize`), потом клиент говорит «понял, начинаем» (`notifications/initialized`), и только после этого можно что-то просить.

In [7]:
async with connect(mcp, show_handshake=True) as session:
    result = await session.list_tools()

print("\nинструменты:", [tool.name for tool in result.tools])

→ клиент: {"method":"initialize","params":{"protocolVersion":"2025-11-25","capabilities":{},"clientInfo":{"name":"mcp","version":"0.1.0"}},"jsonrpc":"2.0","id":0}
← сервер: {"jsonrpc":"2.0","id":0,"result":{"protocolVersion":"2025-11-25","capabilities":{"experimental":{},"prompts":{"listChanged":false},"resources":{"subscribe":false,"listChanged":false},"tools":{"listChanged":false}},"serverInfo":{"name":"demo","version":"1.30.0"}}}
→ клиент: {"method":"notifications/initialized","jsonrpc":"2.0"}
→ клиент: {"method":"tools/list","jsonrpc":"2.0","id":1}
← сервер: {"jsonrpc":"2.0","id":1,"result":{"tools":[{"name":"add","description":"Add two numbers.","inputSchema":{"properties":{"a":{"title":"A","type":"integer"},"b":{"title":"B","type":"integer"}},"required":["a","b"],"title":"addArguments","type":"object"},"outputSchema":{"properties":{"result":{"title":"Result","type":"integer"}},"required":["result"],"title":"addOutput","type":"object"}},{"name":"get_ …

инструменты: ['add', 'get_w

Как читать сообщения:

- `"jsonrpc": "2.0"` — формат сообщений (JSON-RPC), всегда одинаковый;
- `"method"` — что просим: `initialize`, `tools/list`, `tools/call`…;
- `"id"` — номер запроса: ответ приходит с тем же номером, так их и сопоставляют;
- у `notifications/initialized` **нет `id`** — это уведомление, ответа на него не бывает;
- в ответе на `initialize` сервер рассказывает о себе: имя `demo` и что у него есть `tools`, `resources`, `prompts`.

Дальше рукопожатие печатать не будем — оно всегда одно и то же.

### Вызов инструмента — `tools/call`

Именно это происходит, когда модель решает «вызову-ка я `add`».

In [8]:
async with connect(mcp) as session:
    await session.list_tools()                                  # 1) what can you do?
    result = await session.call_tool("add", {"a": 2, "b": 3})   # 2) do it

print("\nответ:", result.content[0].text)

→ клиент: {"method":"tools/list","jsonrpc":"2.0","id":1}
← сервер: {"jsonrpc":"2.0","id":1,"result":{"tools":[{"name":"add","description":"Add two numbers.","inputSchema":{"properties":{"a":{"title":"A","type":"integer"},"b":{"title":"B","type":"integer"}},"required":["a","b"],"title":"addArguments","type":"object"},"outputSchema":{"properties":{"result":{"title":"Result","type":"integer"}},"required":["result"],"title":"addOutput","type":"object"}},{"name":"get_ …
→ клиент: {"method":"tools/call","params":{"name":"add","arguments":{"a":2,"b":3}},"jsonrpc":"2.0","id":2}
← сервер: {"jsonrpc":"2.0","id":2,"result":{"content":[{"type":"text","text":"5"}],"structuredContent":{"result":5},"isError":false}}

ответ: 5


Как и любой агент, клиент сначала узнаёт список инструментов (`tools/list`), потом вызывает нужный: `tools/call` с именем и аргументами → ответ `content` с текстом `"5"` (и то же число в `structuredContent` — для программ).

### Ошибка — тоже обычный ответ

Передадим строку вместо числа. Сервер не падает: он проверяет аргументы по схеме и возвращает ответ с `isError: true` и понятным текстом — модель прочитает его и исправится.

In [9]:
async with connect(mcp) as session:
    await session.list_tools()
    result = await session.call_tool("add", {"a": "два", "b": 3})

print("\nошибка?", result.isError)
print(result.content[0].text.splitlines()[0])

→ клиент: {"method":"tools/list","jsonrpc":"2.0","id":1}
← сервер: {"jsonrpc":"2.0","id":1,"result":{"tools":[{"name":"add","description":"Add two numbers.","inputSchema":{"properties":{"a":{"title":"A","type":"integer"},"b":{"title":"B","type":"integer"}},"required":["a","b"],"title":"addArguments","type":"object"},"outputSchema":{"properties":{"result":{"title":"Result","type":"integer"}},"required":["result"],"title":"addOutput","type":"object"}},{"name":"get_ …
→ клиент: {"method":"tools/call","params":{"name":"add","arguments":{"a":"два","b":3}},"jsonrpc":"2.0","id":2}
← сервер: {"jsonrpc":"2.0","id":2,"result":{"content":[{"type":"text","text":"Error executing tool add: 1 validation error for addArguments\na\n  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='два', input_type=str]\n    For further information visit https://errors.pydantic.dev/2.13/v/int_parsing"}],"isError":true}}

ошибка? True
Error executing tool add: 1 valid

### Ресурс и промпт

In [10]:
async with connect(mcp) as session:
    resource = await session.read_resource("demo://about")
    prompt = await session.get_prompt("trip", {"city": "Прага"})

print("\nресурс:", resource.contents[0].text)
print("промпт:", prompt.messages[0].content.text)

→ клиент: {"method":"resources/read","params":{"uri":"demo://about"},"jsonrpc":"2.0","id":1}
← сервер: {"jsonrpc":"2.0","id":1,"result":{"contents":[{"uri":"demo://about","mimeType":"text/plain","text":"Учебный MCP-сервер: складывает числа и сочиняет погоду."}]}}
→ клиент: {"method":"prompts/get","params":{"name":"trip","arguments":{"city":"Прага"}},"jsonrpc":"2.0","id":2}
← сервер: {"jsonrpc":"2.0","id":2,"result":{"description":"Plan a weekend trip.","messages":[{"role":"user","content":{"type":"text","text":"Спланируй выходные в городе Прага. Сначала узнай погоду."}}]}}

ресурс: Учебный MCP-сервер: складывает числа и сочиняет погоду.
промпт: Спланируй выходные в городе Прага. Сначала узнай погоду.


## 4. В жизни сервер — отдельная программа

Положим тот же сервер в файл. Единственное новое — `mcp.run()` в конце: сервер читает запросы из stdin и пишет ответы в stdout (транспорт **stdio**). Так Claude Desktop и Cursor запускают локальные MCP-серверы.

In [11]:
%%writefile hello_server.py
"""The demo MCP server from 00_mcp_hello.ipynb as a standalone program (stdio)."""

from mcp.server.fastmcp import FastMCP

mcp = FastMCP("demo", log_level="WARNING")


@mcp.tool()
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b


@mcp.tool()
def get_weather(city: str) -> str:
    """Current weather in a city (fake data)."""
    return f"В городе {city} +18°C, солнечно"


if __name__ == "__main__":
    mcp.run()   # stdio: requests come on stdin, answers go to stdout

Overwriting hello_server.py


Клиентский код — тот же `ClientSession`, поменялся только «провод»: теперь клиент сам запускает сервер как процесс и говорит с ним через stdin/stdout.

In [12]:
import sys

from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

server = StdioServerParameters(command=sys.executable, args=["hello_server.py"])

async with stdio_client(server) as (read, write), ClientSession(read, write) as session:
    await session.initialize()
    tools = await session.list_tools()
    weather = await session.call_tool("get_weather", {"city": "Брно"})

print([tool.name for tool in tools.tools])
print(weather.content[0].text)

['add', 'get_weather']
В городе Брно +18°C, солнечно


## Итоги

- MCP-сервер = функции с `@mcp.tool()` / `@mcp.resource()` / `@mcp.prompt()`; протокол делает FastMCP;
- модель видит не код, а **описание**: имя, docstring и JSON-схему аргументов;
- клиент и сервер обмениваются JSON-сообщениями: `initialize` → `tools/list` → `tools/call`…;
- ошибки — это обычные ответы с `isError: true`, сервер не падает;
- «провод» бывает разный (память, stdio, HTTP), код клиента от этого почти не меняется.

Дальше, в `01_mcp_server_client`, — сервер поверх настоящей банковской базы в Postgres.